Propósito: Filtra los tickers propuestos para traerlos filtrados por condiciones.

Usa parametros del indicador RSI de 60min, 1d y 7d.

Este filtro busca capturar movimientos de la siguiente forma: Activo sobrevendido en 1d y 60m; la temporalidad baja indica el movimiento de reversion alcista en conjunto con una temporalidad diaria subiendo. La temporalidad más larga indica la tendencia del activo en largo plazo.

In [15]:
# Instala las librerias requeridas.

%pip install -q pandas-ta yfinance lxml

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [28]:
# Importa las librerias requeridas.
import pandas as pd
import requests
import io
import yfinance as yf
import pandas_ta as ta
import matplotlib.pyplot as plt

In [17]:
#Scrapea Wikipedia para acceder a los nombres de los tickers del S&P500.

#1.1 Define la URL y un 'User-Agent' para evitar el bloqueo.
url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
header = {
  "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/50.0.2661.75 Safari/537.36",
  "X-Requested-With": "XMLHttpRequest"
}

#1.2 Realiza la petición con el header de seguridad
r = requests.get(url, headers=header)

#1.3 Lee la tabla desde el texto de la respuesta, usando io.StringIO para forzar la interpretación como string HTML
tables = pd.read_html(io.StringIO(r.text))
df_spy = tables[0]

#1.4 Extrae la lista de Tickers
tickers_a_analizar = [t.replace('.', '-') for t in df_spy['Symbol'].tolist()]

#1.5 Añade tickers de interes.

mis_tickers = ['^VIX', 'SPY', 'SPXL' , 'TQQQ' , 'QQQ' , 'URA' , 'XLI' , 'XLF' , 'XLRE' , 'XLB' , 'XLY' , 'XLE', 'XLV' ,
                'XBI' , 'XLC' , 'BTC-USD' , 'DIA']

tickers_a_analizar.extend(mis_tickers)

if tickers_a_analizar:
  print(f"Se han cargado {len(tickers_a_analizar)} tickers...")

Se han cargado 520 tickers...


In [34]:
#ANALISIS TEMPORALIDAD 60MIN.

empresas_60min = []

for ticker in tickers_limpios:
    try:
        df = data_60m[ticker].dropna()
        if len(df) < 50: continue

        # Cálculos Técnicos
        df.ta.rsi(length=10, append=True)
        df['RSI_MA'] = ta.sma(df['RSI_10'], length=14)
        df.ta.macd(fast=12, slow=26, signal=9, append=True)

        # Variables para validación.
        rsi_act = df['RSI_10'].iloc[-1]
        rsi_ant = df['RSI_10'].iloc[-2]
        macd_act = df['MACD_12_26_9'].iloc[-1]
        sig_act = df['MACDs_12_26_9'].iloc[-1]

        # Lógica de Filtrado 60min
        if (rsi_act <= 35 
            and rsi_act >= rsi_ant
                ):
            
            empresas_60min.append({
                'Ticker': ticker,
                'RSI actual': round(rsi_act, 1),
                'RSI anterior': round(rsi_ant, 1),
                #Añade el MACD.
                'MACD': round(macd_act, 3)
                
            })
    except Exception as e:
      print(f"Error en {ticker}: {e}")

print("\n" + "="*95)
if empresas_60min:
    df_final = pd.DataFrame(empresas_60min)
    print(df_final.sort_values(by='RSI actual').to_string(index=False))
else:
    print("Sin señales detectadas.")


Ticker  RSI actual  RSI anterior    MACD
  KLAC        19.9          16.7  -5.400
    BE        20.6          19.1 -10.551
   FIX        20.8          19.8 -71.908
   HWM        21.1          20.3  -7.170
    GL        23.6          22.3  -3.055
 GOOGL        23.9          21.9  -4.965
   AXP        24.7          22.6  -1.004
   CMG        25.0          22.3  -0.527
  LRCX        25.6          23.9 -10.624
  GOOG        25.8          23.2  -4.740
   LVS        25.8          19.3  -0.569
   CCL        27.3          26.4  -0.182
   XLC        27.6          25.4  -1.033
  TMUS        28.1          27.5  -1.608
   CAT        28.3          27.1 -12.460
  AMAT        28.3          27.9 -10.254
    PH        28.4          28.2 -20.503
  CBRE        29.0          26.5  -2.183
   EME        29.4          27.1 -16.314
  MPWR        29.4          24.7 -28.690
   MET        29.8          22.7  -0.324
   MGM        29.9          28.5  -0.120
  NCLH        30.7          30.4  -0.143
  WYNN        3

In [36]:
# ANÁLISIS DIARIO

empresas_diario = []

for ticker in tickers_limpios:
    try:
        df = data_1d[ticker].dropna()
        if len(df) < 30: continue

        # Cálculos Técnicos.
        df.ta.rsi(length=14, append=True)
        df.ta.macd(fast=12, slow=26, signal=9, append=True)

        # Variables de validación
        rsi_ant = df['RSI_14'].iloc[-2]
        rsi_act = df['RSI_14'].iloc[-1]
        macd_act = df['MACD_12_26_9'].iloc[-1]
        sig_act = df['MACDs_12_26_9'].iloc[-1]
        

        precio_act = df['Close'].iloc[-1]

        # LÓGICA DE FILTRADO DIARIO
        estuvo_sobrevendido = rsi_act <= 35 or rsi_ant <= 35
        dejo_de_caer = rsi_act >= rsi_ant

        if dejo_de_caer:
            
            empresas_diario.append({
                'Ticker': ticker,
                #'Prec diario': round(precio_act, 2),
                'RSI actual': round(rsi_act, 1),
                'RSI anterior': round(rsi_ant, 1),
                'MACD': round(macd_act, 3),
                #'MACD_SIGNAL': round(sig_act , 3)
            })
    except:
        continue

print("\n" + "="*95)


print("PAPELES DETECTADOS EN DIARIO:")
if empresas_diario:
    df_final = pd.DataFrame(empresas_diario)
    print(df_final.sort_values(by='RSI actual').to_string(index=False))
else:
    print("Sin señales diarias.")


PAPELES DETECTADOS EN DIARIO:
 Ticker  RSI actual  RSI anterior     MACD
    ALB        22.2          18.4   -6.826
    MOS        22.5          20.1   -1.631
   LRCX        25.6          25.0  -16.935
   AMAT        26.0          23.9  -22.572
    PWR        28.1          27.5  -16.030
   DXCM        29.6          29.3   -1.682
    GEV        30.2          29.9  -21.866
   SNDK        30.3          29.4  -90.699
   INTC        32.6          30.4   -4.152
   COHR        33.0          32.0  -20.786
      Q        33.0          31.3   -2.683
    TER        34.1          32.6  -14.886
    GLW        34.7          33.4   -6.995
   ANET        35.0          33.2   -6.158
    APH        35.1          31.3   -1.502
   AVGO        35.5          34.4   -7.997
    STX        35.5          35.0  -51.222
    FCX        37.1          34.3   -1.655
   TQQQ        37.5          36.7   -1.547
   FSLR        37.8          37.3   -6.080
   LITE        38.5          37.2  -42.723
   KEYS        38.7    

In [37]:
#TICKERS REPETIDOS EN AMBAS TEMPORALIDADES.

tickers_60 = [e['Ticker'] for e in empresas_60min]
ticker_1d = [e['Ticker'] for e in empresas_diario]
tickers_comunes = list(set(tickers_60) & set(ticker_1d)) # Interseccion de los elementos en la lista.

print(tickers_comunes)


['GEV', 'GLW', 'PWR', 'CIEN', 'LRCX', 'TQQQ', 'AMAT', 'SNDK', 'COHR']


In [39]:
# ANÁLISIS SEMANAL

empresas_semanal = []

for ticker in tickers_limpios:
    try:
        df = data_7d[ticker].dropna()
        if len(df) < 50: continue

        # Cálculos Técnicos.
        df.ta.rsi(length=10, append=True)
        df.ta.macd(fast=12, slow=26, signal=9, append=True)

        # Variables de validación
        rsi_act = df['RSI_10'].iloc[-1]
        macd_act = df['MACD_12_26_9'].iloc[-1]
        sig_act = df['MACDs_12_26_9'].iloc[-1]
        rsi_ant = df['RSI_10'].iloc[-2]

        # Lógica de filtrado.
        if rsi_act <= 44 and (rsi_act > rsi_ant):
        
            empresas_semanal.append({
                'Ticker': ticker,
                'RSI actual': round(rsi_act, 1),
                'RSI anterior': round(rsi_ant, 1),
                'MACD': round(macd_act, 3),
            })
    except:
        continue

print("\n" + "="*95)
print("ACTIVOS DETECTADOS EN SEMANAL.")
if empresas_semanal:
    df_final = pd.DataFrame(empresas_semanal)
    print(df_final.sort_values(by='RSI actual').to_string(index=False))
else:
    print("Sin señales semanales.")


ACTIVOS DETECTADOS EN SEMANAL.
Ticker  RSI actual  RSI anterior     MACD
    GS        27.4          27.2  -12.282
   PLD        29.3          28.1   -2.074
   PNR        30.8          24.0   -8.280
   CRH        34.0          29.3   -7.248
  SKYD        36.0          26.1   -0.663
   ARE        36.4          32.6   -1.109
  VICI        36.6          35.8   -0.975
  LULU        36.6          32.8  -13.504
   WDC        37.0          34.6   -5.314
   LII        37.1          28.8  -38.828
   TXT        37.3          34.5   -3.763
   UAL        37.5          34.4   -1.620
   STX        37.6          37.1   27.856
  AKAM        37.9          32.7   -3.681
   APO        38.1          35.5   -3.049
   ROL        38.4          32.6   -5.069
   FRT        38.6          30.1   -0.739
   EIX        38.6          35.3   -3.238
  FICO        38.7          35.1 -146.279
  CHRW        39.3          38.7   -7.829
     O        39.3          36.4   -1.244
  TTWO        39.9          39.0   -6.553
  

In [40]:
#TICKERS REPETIDOS EN TRIPLE PANTALLA.

tickers_60 = [e['Ticker'] for e in empresas_60min]
tickers_1d = [e['Ticker'] for e in empresas_diario]
#tickers_7d = [e['Ticker'] for e in empresas_semanal]

tickers_comunes = list(set(tickers_60) & set(tickers_1d) 
                       #& set(tickers_7d)
                       )

if tickers_comunes:
    print(tickers_comunes)

['GEV', 'GLW', 'PWR', 'CIEN', 'LRCX', 'TQQQ', 'AMAT', 'SNDK', 'COHR']
